In [24]:
# Зареждане на необходимите библиотеки
import torch
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import MessagePassing

In [25]:
from torch_geometric.datasets import Planetoid
# Зареждане на набора от данни
dataset = Planetoid(root='data/Cora', name='Cora')
data = dataset[0]

print(data)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [26]:
# Създаване на собствен MPNN слой
class SimpleMPNN(MessagePassing):

    def __init__(self):
        super().__init__(aggr='mean')
        
    # Функция за генериране на съобщения
    def message(self, x_j):
        return x_j
        
    # Функция за актуализация
    def update(self, aggr_out):
        return aggr_out

    # Изпълнение на Message Passing
    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)


In [27]:
# Създаване на слоя
layer = SimpleMPNN()

# Изчисляване на новите представяния
new_x = layer(data.x.float(), data.edge_index)

print(new_x.shape)

torch.Size([2708, 1433])


In [38]:
# Проверка на първия възел
diff = data.x[0] != new_x[0]
indices = torch.where(diff)[0]

print(f"Брой променени характеристики: {len(indices)}")
print("Първите 3 променени характеристики:")

for i in indices[:3]:
    print(f"Характеристика {i.item()}: "
          f"{data.x[0, i].item():.2f} -> "
          f"{new_x[0, i].item():.2f}")

Брой променени характеристики: 49
Първите 3 променени характеристики:
Характеристика 41: 0.00 -> 0.33
Характеристика 52: 0.00 -> 0.33
Характеристика 81: 1.00 -> 0.00


In [39]:
# Добавяне на линейна трансформация към функцията за генериране на съобщения.
from torch import nn

class LinearMPNN(MessagePassing):

    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='mean')

        self.lin = nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)

    def message(self, x_j):
        return self.lin(x_j)

    def update(self, aggr_out):
        return F.relu(aggr_out)


In [40]:
# Използване на слоя 
layer = LinearMPNN(dataset.num_node_features, 64)

new_x = layer(data.x.float(), data.edge_index)

print(x.shape)

torch.Size([2708, 64])


In [23]:
# Проверка на първия възел
print("Преди:")
print(data.x[0][:20])

print()

print("След:")
print(new_x[0][:20])

Преди:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 1.])

След:
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0951, 0.0413, 0.1009, 0.0637, 0.0000,
        0.0517, 0.0278, 0.0000, 0.0673, 0.0085, 0.0105, 0.0164, 0.0426, 0.0584,
        0.0000, 0.0050], grad_fn=<SliceBackward0>)
